# Learned against exact equaliser

Uncoded BER with the network in place of the BCJR. Both curves come from the same simulator, so the only difference is the equaliser.

In [ ]:
import subprocess, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
MSPRS = ROOT / "build" / "bin" / "msprs"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "legend.fontsize": 7.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.4, "axes.grid": True,
    "grid.alpha": 0.3, "grid.linestyle": ":", "figure.dpi": 110,
    "savefig.dpi": 200, "savefig.bbox": "tight", "pdf.fonttype": 42,
})

def ber(*args):
    out = subprocess.run([str(MSPRS), *map(str, args)], capture_output=True,
                         text=True, check=True).stdout
    rows = [l.split("|") for l in out.splitlines() if l and not l.startswith("#")]
    return (np.array([float(r[0]) for r in rows]),
            np.array([float(r[-1]) for r in rows]))

In [ ]:
GRID = ["--ebn0-min", 2, "--ebn0-max", 10.01, "--ebn0-step", 1, "--seed", 3]

x_l, y_l = ber("--mode", "siso-ber", "--L0", 3, "--family", "balanced",
               "--siso", ROOT / "siso_nn" / "weights.txt", "--frames", 20, *GRID)
x_e, y_e = ber("--mode", "uncoded-msprs", "--L0", 3, "--family", "balanced",
               "--be", 10**9, "--min-fra", 20, "--max-fra", 20, "--threads", 1, *GRID)

fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.semilogy(x_e, y_e, "o-", color="#000000", label="exact BCJR")
ax.semilogy(x_l, y_l, "s--", color="#d62728", label="learned")
ax.set(xlabel="$E_b/N_0$ (dB)", ylabel="BER", ylim=(1e-4, 0.5))
ax.legend()
for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"siso_learned.{ext}")

for a, b, c in zip(x_e, y_e, y_l):
    print(f"  {a:4.0f} dB   exact {b:.3e}   learned {c:.3e}   {c/b:6.1f}x")